# Comodity.

In [3]:
import os
import pandas as pd

# Lists the files to view the naming pattern
archivos = os.listdir('../data/raw/Salient_Commodity_Data_Release_Grouped_MCS_2025')
print(archivos[:10])  # First 10 csv.
print(f"Total files: {len(archivos)}")

['mcs2025-abras_meta.xml', 'mcs2025-abras_salient.csv', 'mcs2025-alumi_meta.xml', 'mcs2025-alumi_salient.csv', 'mcs2025-antim_meta.xml', 'mcs2025-antim_salient.csv', 'mcs2025-arsen_meta.xml', 'mcs2025-arsen_salient.csv', 'mcs2025-asbes_meta.xml', 'mcs2025-asbes_salient.csv']
Total files: 170


In [4]:
df_ejemplo = pd.read_csv('../data/raw/Salient_Commodity_Data_Release_Grouped_MCS_2025/mcs2025-fluor_salient.csv')
print(df_ejemplo.columns.tolist())
df_ejemplo.head()

['DataSource', 'Commodity', 'Year', 'USprod_Metallurgical_kt', 'USprod_H2SiF6_kt', 'Imports_Acid_kt', 'Imports_Metallurgical_kt', 'Imports_Total_Fl_kt', 'Imports_HF_kt', 'Imports_AlF3_kt', 'Imports_Cryolite_kt', 'Exports_All_grades_kt', 'Consump_Apprnt_kt', 'Price_Acid_dt', 'Price_Metallurgical_dt', 'Employment_Mine_num', 'NIR_pct']


,DataSource,Commodity,Year,USprod_Metallurgical_kt,USprod_H2SiF6_kt,Imports_Acid_kt,Imports_Metallurgical_kt,Imports_Total_Fl_kt,Imports_HF_kt,Imports_AlF3_kt,Imports_Cryolite_kt,Exports_All_grades_kt,Consump_Apprnt_kt,Price_Acid_dt,Price_Metallurgical_dt,Employment_Mine_num,NIR_pct
0,MCS2025,Fluorspar,2020,NaN,22,427,65,492,103,21,26,9,483,309,149,16,100
1,MCS2025,Fluorspar,2021,NaN,40,391,59,451,103,28,42,15,436,322,151,17,100
2,MCS2025,Fluorspar,2022,NaN,43,448,84,532,99,21,28,24,508,387,206,15,100
3,MCS2025,Fluorspar,2023,NaN,43,381,31,412,87,25,32,20,392,429,296,16,100
4,MCS2025,Fluorspar,2024,NaN,40,400,40,440,75,24,24,15,430,470,400,15,100


In [5]:
file = '../data/raw/Salient_Commodity_Data_Release_Grouped_MCS_2025'
archivos_csv = [f for f in os.listdir(file) if f.endswith('_salient.csv')]

resumen_precios = []

for archivo in archivos_csv:
    try:
        df_temp = pd.read_csv(os.path.join(file, archivo))
        commodity = df_temp['Commodity'].iloc[0] if 'Commodity' in df_temp.columns else None
        columnas_precio = [c for c in df_temp.columns if c.startswith('Price_')]
        resumen_precios.append({
            'archivo': archivo,
            'commodity': commodity,
            'columnas_precio': columnas_precio
        })
    except Exception as e:
        resumen_precios.append({'archivo': archivo, 'commodity': None, 'columnas_precio': f'ERROR: {e}'})

df_resumen = pd.DataFrame(resumen_precios)
pd.set_option('display.max_colwidth', None)
print(df_resumen[df_resumen['commodity'].notna()].to_string())

                        archivo                       commodity                                                                                                                                                                                                           columnas_precio
0     mcs2025-abras_salient.csv        Abrasives (Manufactured)                                                                                                                                                  [Price_Al2O3_dt, Price_Al2O3_HiPur_dt, Price_SiC_dt, Price_Met_Abras_dt]
1     mcs2025-alumi_salient.csv                        Aluminum                                                                                                                                                                                                       [Price_Ingot_ctslb]
2     mcs2025-antim_salient.csv                        Antimony                                                                                           

In [6]:
df_titanio = pd.read_csv(f'{file}/mcs2025-timin_salient.csv')
print(df_titanio.columns.tolist())
df_titanio.head(3)

['DataSource', 'Commodity', 'Year', 'USprod_kt', 'Imports_kt', 'Exports_kt', 'Consump_kt', 'Price_dt', 'Price_dt.1', 'Price_dt.2', 'Price_dt.3', 'Employment_num', 'NIR_pct']


,DataSource,Commodity,Year,USprod_kt,Imports_kt,Exports_kt,Consump_kt,Price_dt,Price_dt.1,Price_dt.2,Price_dt.3,Employment_num,NIR_pct
0,MCS2025,Titanium Mineral Concentrates,2020,100,807,18,900,1170,459,215,757,315,89
1,MCS2025,Titanium Mineral Concentrates,2021,100,969,30,1000,1300,595,240,774,290,90
2,MCS2025,Titanium Mineral Concentrates,2022,200,952,110,1000,1470,530,285,867,390,81


In [7]:
with open(f'{file}/mcs2025-timin_meta.xml', 'r', encoding='utf-8') as f:
    contenido = f.read()

# Busca las menciones de "Price" en el XML para ver el orden y la etiqueta real de cada columna
import re
matches = re.findall(r'<attrlabl>(Price.*?)</attrlabl>.*?<attrdef>(.*?)</attrdef>', contenido, re.DOTALL)
for etiqueta, definicion in matches:
    print(etiqueta, '→', definicion[:150])

Price_Rutile_dt → Price, dollars per metric ton: Rutile, bulk, minimum 95% TiO2, free on board (f.o.b.) Australia. Source: Fast Markets IM; average of yearend price. Da
Price_Ilmente_Leucoxene_dt → Price, dollars per metric ton: Ilmenite and leucoxene, bulk, f.o.b. Australia. Source: Zen Innovations AG, Global Trade Tracker. Data are estimated fo
Price_Ilmenite_dt → Price, dollars per metric ton: Ilmenite, average unit value of imports. Landed duty-paid unit value based on U.S. imports for consumption. Source: U.S
Price_Slag_dt → Price, dollars per metric ton: Slag, 80%–95% TiO2, average unit value of imports. Landed duty-paid unit value based on U.S. imports for consumption. S


In [8]:
import sys
sys.path.append('..')
from etl.db import get_engine

engine = get_engine()
carpeta = '../data/raw/Salient_Commodity_Data_Release_Grouped_MCS_2025'

In [9]:
price_mapping = {
    'aluminum': ('mcs2025-alumi_salient.csv', ['Price_Ingot_ctslb'], 'first'),
    'antimony': ('mcs2025-antim_salient.csv', ['Price_Metal_dlb'], 'first'),
    'arsenic': ('mcs2025-arsen_salient.csv', ['Price_Mtl_US_dkg'], 'first'),
    'barite': ('mcs2025-barit_salient.csv', ['Price_dt'], 'first'),
    'beryllium': ('mcs2025-beryl_salient.csv', ['Price_Alloy_dkg'], 'first'),
    'bismuth': ('mcs2025-bismu_salient.csv', ['Price_Average_dlb'], 'first'),
    'boron': ('mcs2025-boron_salient.csv', ['Price_CIF_dt'], 'first'),
    'chromium': ('mcs2025-chrom_salient.csv', ['Price_Ferrochromium_dlb'], 'first'),
    'cobalt': ('mcs2025-cobal_salient.csv', ['Price_LME_dlb'], 'first'),
    'copper': ('mcs2025-coppe_salient.csv', ['Price_LME_ctslb'], 'first'),
    'fluorspar': ('mcs2025-fluor_salient.csv', ['Price_Acid_dt'], 'first'),
    'gallium': ('mcs2025-galli_salient.csv', ['Price_High-purity_dkg'], 'first'),
    'germanium': ('mcs2025-germa_salient.csv', ['Price_Metal_dkg'], 'first'),
    'graphite': ('mcs2025-graph_salient.csv', ['Price_Flake_dt'], 'first'),
    'indium': ('mcs2025-indiu_salient.csv', ['Price_NY_dkg'], 'first'),
    'lead': ('mcs2025-lead_salient.csv', ['Price_LME_ctslb'], 'first'),
    'lithium': ('mcs2025-lithium_salient.csv', ['Price_dt'], 'first'),
    'magnesium': ('mcs2025-mgmet_salient.csv', ['Price_dt'], 'first'),
    'manganese': ('mcs2025-manga_salient.csv', ['Price_CN_CIF_dt'], 'first'),
    'nickel': ('mcs2025-nicke_salient.csv', ['Price_dt'], 'first'),
    'niobium': ('mcs2025-niobi_salient.csv', ['Price_dkg'], 'first'),
    'phosphate': ('mcs2025-phosp_salient.csv', ['Price_dt'], 'first'),
    'potash': ('mcs2025-potas_salient.csv', ['Price_Muriate_dt'], 'first'),
    'rhenium': ('mcs2025-rheni_salient.csv', ['Price_Metal_dkg'], 'first'),
    'silicon': ('mcs2025-simet_salient.csv', ['Price_Si_ctslb'], 'first'),
    'silver': ('mcs2025-silve_salient.csv', ['Price_Bullion_dtoz'], 'first'),
    'tantalum': ('mcs2025-tanta_salient.csv', ['Price_Ta2O5_dkg'], 'first'),
    'tellurium': ('mcs2025-tellu_salient.csv', ['Price_US_dkg'], 'first'),
    'tin': ('mcs2025-tin_salient.csv', ['Price_LME_ctslb'], 'first'),
    'titanium': ('mcs2025-timin_salient.csv', ['Price_dt', 'Price_dt.2'], 'mean'),
    'tungsten': ('mcs2025-tungs_salient.csv', ['Price_WO3_dt'], 'first'),
    'vanadium': ('mcs2025-vanad_salient.csv', ['Price_V2O5_dlb'], 'first'),
    'zinc': ('mcs2025-zinc_salient.csv', ['Price_LME_ctslb'], 'first'),
    'zirconium': ('mcs2025-zirco_salient.csv', ['Price_Zircon_imported_t'], 'first'),
    'hafnium': ('mcs2025-zirco_salient.csv', ['Price_Hf_unwrought_t'], 'first'),
    'platinum': ('mcs2025-plati_salient.csv', ['Price_Platinum_dto'], 'first'),
    'palladium': ('mcs2025-plati_salient.csv', ['Price_Palladium_dto'], 'first'),
    'iridium': ('mcs2025-plati_salient.csv', ['Price_Iridium_dto'], 'first'),
    'rhodium': ('mcs2025-plati_salient.csv', ['Price_Rhodium_dto'], 'first'),
    'ruthenium': ('mcs2025-plati_salient.csv', ['Price_Ruthenium_dto'], 'first'),
    'cerium': ('mcs2025-rareee_salient.csv', ['Price_CeO2_dkg'], 'first'),
    'dysprosium': ('mcs2025-rareee_salient.csv', ['Price_Dy2O3_dkg'], 'first'),
    'europium': ('mcs2025-rareee_salient.csv', ['Price_Eu2O3_dkg'], 'first'),
    'lanthanum': ('mcs2025-rareee_salient.csv', ['Price_La2O3_dkg'], 'first'),
    'neodymium': ('mcs2025-rareee_salient.csv', ['Price_Nd2O3_dkg'], 'first'),
    'terbium': ('mcs2025-rareee_salient.csv', ['Price_TbO2_dkg'], 'first'),
    'erbium': ('mcs2025-rareee_salient.csv', ['Price_Mischmetal_dkg'], 'first'),
    'gadolinium': ('mcs2025-rareee_salient.csv', ['Price_Mischmetal_dkg'], 'first'),
    'holmium': ('mcs2025-rareee_salient.csv', ['Price_Mischmetal_dkg'], 'first'),
    'lutetium': ('mcs2025-rareee_salient.csv', ['Price_Mischmetal_dkg'], 'first'),
    'praseodymium': ('mcs2025-rareee_salient.csv', ['Price_Mischmetal_dkg'], 'first'),
    'samarium': ('mcs2025-rareee_salient.csv', ['Price_Mischmetal_dkg'], 'first'),
    'thulium': ('mcs2025-rareee_salient.csv', ['Price_Mischmetal_dkg'], 'first'),
    'ytterbium': ('mcs2025-rareee_salient.csv', ['Price_Mischmetal_dkg'], 'first'),
    'yttrium': ('mcs2025-yttri_salient.csv', ['Price_Y2O3_dkg'], 'first'),
}

print(f"Maping minerals: {len(price_mapping)} from 55")

Maping minerals: 55 from 55


In [10]:
def get_unit_factor(nombre_columna):
    if nombre_columna.endswith('_dt') or nombre_columna.endswith('_t') or 'dt.' in nombre_columna:
        return 1
    elif nombre_columna.endswith('_dlb'):
        return 2204.62
    elif nombre_columna.endswith('_ctslb'):
        return 2204.62 / 100
    elif nombre_columna.endswith('_dkg'):
        return 1000
    elif nombre_columna.endswith('_dto') or nombre_columna.endswith('_dtoz'):
        return 32150.7
    elif nombre_columna.endswith('_dg'):
        return 1_000_000
    else:
        return None

In [11]:
price_rows = []
is_proxy_set = {'erbium', 'gadolinium', 'holmium', 'lutetium', 'praseodymium', 'samarium', 'thulium', 'ytterbium'}

for mineral_name, (archivo, columnas, metodo) in price_mapping.items():
    df_precio = pd.read_csv(f'{carpeta}/{archivo}')
    df_precio['Year'] = df_precio['Year'].astype(int)

    for _, row in df_precio.iterrows():
        valores_normalizados = []
        for c in columnas:
            if pd.notna(row.get(c)):
                factor = get_unit_factor(c)
                if factor is None:
                    print(f"AVISO: sufijo no reconocido en columna {c} (mineral: {mineral_name})")
                    continue
                valores_normalizados.append(row[c] * factor)

        if not valores_normalizados:
            continue

        precio_por_tonelada = sum(valores_normalizados) / len(valores_normalizados)

        price_rows.append({
            'mineral_name': mineral_name,
            'year': row['Year'],
            'price_per_tonne': round(precio_por_tonelada, 2),
            'is_proxy': mineral_name in is_proxy_set,
            'source': f'USGS MCS 2025 ({archivo})'
        })

df_prices_final = pd.DataFrame(price_rows)
print(df_prices_final.shape)
df_prices_final.sort_values('price_per_tonne', ascending=False).head(10)

(271, 5)


,mineral_name,year,price_per_tonne,is_proxy,source
187,rhodium,2021,6.511835e+08,False,USGS MCS 2025 (mcs2025-plati_salient.csv)
188,rhodium,2022,5.010687e+08,False,USGS MCS 2025 (mcs2025-plati_salient.csv)
186,rhodium,2020,3.602505e+08,False,USGS MCS 2025 (mcs2025-plati_salient.csv)
189,rhodium,2023,2.141423e+08,False,USGS MCS 2025 (mcs2025-plati_salient.csv)
182,iridium,2021,1.658462e+08,False,USGS MCS 2025 (mcs2025-plati_salient.csv)
185,iridium,2024,1.543234e+08,False,USGS MCS 2025 (mcs2025-plati_salient.csv)
184,iridium,2023,1.502331e+08,False,USGS MCS 2025 (mcs2025-plati_salient.csv)
190,rhodium,2024,1.478932e+08,False,USGS MCS 2025 (mcs2025-plati_salient.csv)
183,iridium,2022,1.473123e+08,False,USGS MCS 2025 (mcs2025-plati_salient.csv)
177,palladium,2021,7.777833e+07,False,USGS MCS 2025 (mcs2025-plati_salient.csv)


In [12]:
df_prices_final.sort_values('price_per_tonne', ascending=True).head(10)

,mineral_name,year,price_per_tonne,is_proxy,source
86,manganese,2020,4.58,False,USGS MCS 2025 (mcs2025-manga_salient.csv)
89,manganese,2023,4.80,False,USGS MCS 2025 (mcs2025-manga_salient.csv)
87,manganese,2021,5.27,False,USGS MCS 2025 (mcs2025-manga_salient.csv)
90,manganese,2024,5.80,False,USGS MCS 2025 (mcs2025-manga_salient.csv)
88,manganese,2022,5.97,False,USGS MCS 2025 (mcs2025-manga_salient.csv)
101,phosphate,2020,76.00,False,USGS MCS 2025 (mcs2025-phosp_salient.csv)
102,phosphate,2021,83.00,False,USGS MCS 2025 (mcs2025-phosp_salient.csv)
103,phosphate,2022,99.00,False,USGS MCS 2025 (mcs2025-phosp_salient.csv)
105,phosphate,2024,100.00,False,USGS MCS 2025 (mcs2025-phosp_salient.csv)
104,phosphate,2023,101.00,False,USGS MCS 2025 (mcs2025-phosp_salient.csv)


In [13]:
# change price "dollars per metric ton unit" to "per metric ton".

df_prices_final.loc[df_prices_final['mineral_name'] == 'manganese', 'price_per_tonne'] *= 44

print(df_prices_final[df_prices_final['mineral_name'] == 'manganese'])

   mineral_name  year  price_per_tonne  is_proxy  \
86    manganese  2020           201.52     False   
87    manganese  2021           231.88     False   
88    manganese  2022           262.68     False   
89    manganese  2023           211.20     False   
90    manganese  2024           255.20     False   

                                       source  
86  USGS MCS 2025 (mcs2025-manga_salient.csv)  
87  USGS MCS 2025 (mcs2025-manga_salient.csv)  
88  USGS MCS 2025 (mcs2025-manga_salient.csv)  
89  USGS MCS 2025 (mcs2025-manga_salient.csv)  
90  USGS MCS 2025 (mcs2025-manga_salient.csv)  


In [14]:
df_prices_final.sort_values('price_per_tonne', ascending=True).head(15)

,mineral_name,year,price_per_tonne,is_proxy,source
101,phosphate,2020,76.00,False,USGS MCS 2025 (mcs2025-phosp_salient.csv)
102,phosphate,2021,83.00,False,USGS MCS 2025 (mcs2025-phosp_salient.csv)
103,phosphate,2022,99.00,False,USGS MCS 2025 (mcs2025-phosp_salient.csv)
105,phosphate,2024,100.00,False,USGS MCS 2025 (mcs2025-phosp_salient.csv)
104,phosphate,2023,101.00,False,USGS MCS 2025 (mcs2025-phosp_salient.csv)
17,barite,2022,145.00,False,USGS MCS 2025 (mcs2025-barit_salient.csv)
16,barite,2021,167.00,False,USGS MCS 2025 (mcs2025-barit_salient.csv)
146,tungsten,2020,172.00,False,USGS MCS 2025 (mcs2025-tungs_salient.csv)
15,barite,2020,183.00,False,USGS MCS 2025 (mcs2025-barit_salient.csv)
86,manganese,2020,201.52,False,USGS MCS 2025 (mcs2025-manga_salient.csv)


In [15]:
with open(f'{file}/mcs2025-tungs_meta.xml', 'r', encoding='utf-8') as f:
    contenido = f.read()

matches = re.findall(r'<attrlabl>(Price.*?)</attrlabl>.*?<attrdef>(.*?)</attrdef>', contenido, re.DOTALL)
for etiqueta, definicion in matches:
    print(etiqueta, '→', definicion[:250])

Price_WO3_dt → Price: Concentrate, average in-warehouse Rotterdam, dollars per dry metric ton unit of tungsten trioxide. Data are estimated for the most recent year. A metric ton unit of tungsten trioxide contains 7.93 kilograms of tungsten. Source: Argus Media Gro


In [16]:
# change price "dollars per metric ton unit" to "per metric ton".

df_prices_final.loc[df_prices_final['mineral_name'] == 'tungsten', 'price_per_tonne'] *= 65

print(df_prices_final[df_prices_final['mineral_name'] == 'tungsten'])

    mineral_name  year  price_per_tonne  is_proxy  \
146     tungsten  2020          11180.0     False   
147     tungsten  2021          14625.0     False   
148     tungsten  2022          17875.0     False   
149     tungsten  2023          16770.0     False   
150     tungsten  2024          16250.0     False   

                                        source  
146  USGS MCS 2025 (mcs2025-tungs_salient.csv)  
147  USGS MCS 2025 (mcs2025-tungs_salient.csv)  
148  USGS MCS 2025 (mcs2025-tungs_salient.csv)  
149  USGS MCS 2025 (mcs2025-tungs_salient.csv)  
150  USGS MCS 2025 (mcs2025-tungs_salient.csv)  


In [17]:
#Checking the whole table.

minerales_con_mtu = []

for mineral_name, (archivo, columnas, metodo) in price_mapping.items():
    meta_file = archivo.replace('_salient.csv', '_meta.xml')
    ruta_meta = f'{carpeta}/{meta_file}'
    if not os.path.exists(ruta_meta):
        continue
    with open(ruta_meta, 'r', encoding='utf-8') as f:
        contenido = f.read()
    if 'metric ton unit' in contenido.lower() or 'dmtu' in contenido.lower():
        minerales_con_mtu.append(mineral_name)

print(set(minerales_con_mtu))

{'tungsten', 'manganese'}


In [18]:
# Last verification.

print("Final Shape:", df_prices_final.shape)
print()
print("Price range (ordered ascending, top 5):")
print(df_prices_final.sort_values('price_per_tonne').head(5)[['mineral_name', 'year', 'price_per_tonne']])
print()
print("Price range (ordered downward, top 5):")
print(df_prices_final.sort_values('price_per_tonne', ascending=False).head(5)[['mineral_name', 'year', 'price_per_tonne']])

Final Shape: (271, 5)

Price range (ordered ascending, top 5):
    mineral_name  year  price_per_tonne
101    phosphate  2020             76.0
102    phosphate  2021             83.0
103    phosphate  2022             99.0
105    phosphate  2024            100.0
104    phosphate  2023            101.0

Price range (ordered downward, top 5):
    mineral_name  year  price_per_tonne
187      rhodium  2021     6.511835e+08
188      rhodium  2022     5.010687e+08
186      rhodium  2020     3.602505e+08
189      rhodium  2023     2.141423e+08
182      iridium  2021     1.658462e+08


In [22]:
df_minerals_db = pd.read_sql('SELECT mineral_id, name FROM minerals', engine)
print(df_minerals_db.shape)
df_minerals_db.head()

(55, 2)


,mineral_id,name
0,1,aluminum
1,2,antimony
2,3,arsenic
3,4,barite
4,5,beryllium


In [23]:
df_prices_ready = df_prices_final.merge(
    df_minerals_db, left_on='mineral_name', right_on='name', how='left'
)

sin_match = df_prices_ready[df_prices_ready['mineral_id'].isna()]
print(f"Filas sin match: {len(sin_match)}")
print(sin_match[['mineral_name']].drop_duplicates())

Filas sin match: 0
Empty DataFrame
Columns: [mineral_name]
Index: []


In [24]:
df_prices_ready['price_date'] = pd.to_datetime(df_prices_ready['year'].astype(str) + '-01-01')
df_prices_ready['unit'] = 'USD/tonne'
df_prices_ready['market_source'] = df_prices_ready['source']
df_prices_ready = df_prices_ready.rename(columns={'price_per_tonne': 'price_usd'})

df_to_insert_prices = df_prices_ready[[
    'mineral_id', 'price_date', 'price_usd', 'unit', 'market_source', 'is_proxy'
]].copy()
df_to_insert_prices['mineral_id'] = df_to_insert_prices['mineral_id'].astype(int)

print(df_to_insert_prices.shape)
df_to_insert_prices.head()

(271, 6)


,mineral_id,price_date,price_usd,unit,market_source,is_proxy
0,1,2020-01-01,1977.54,USD/tonne,USGS MCS 2025 (mcs2025-alumi_salient.csv),False
1,1,2021-01-01,3053.40,USD/tonne,USGS MCS 2025 (mcs2025-alumi_salient.csv),False
2,1,2022-01-01,3364.25,USD/tonne,USGS MCS 2025 (mcs2025-alumi_salient.csv),False
3,1,2023-01-01,2775.62,USD/tonne,USGS MCS 2025 (mcs2025-alumi_salient.csv),False
4,1,2024-01-01,2866.01,USD/tonne,USGS MCS 2025 (mcs2025-alumi_salient.csv),False


In [25]:
existing = pd.read_sql('SELECT COUNT(*) as n FROM prices', engine)
print(f"Row prices: {existing['n'][0]}")

Row prices: 0


In [28]:
df_to_insert_prices.to_sql('prices', engine, if_exists='append', index=False)
print("Load prices")

Load prices
